In [1]:
import numpy as np
# from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd

# Теория

### 1.1 Аналитическое решение линейной регрессии
Мы ищем такие веса, чтобы предсказания были максимально близко к настоящим значениям.
Если записать все объекты в матрицу X, а ответы в вектор y, то решение можно получить напрямую формулой:
"веса = (X транспонированная * X) в обратной * X транспонированная * y".
Идея: это способ найти лучшие веса без итераций (без градиентного спуска).

### 1.2 Что меняется при добавлении L1 и L2 регуляризации
Регуляризация добавляет штраф к функции ошибки, чтобы веса не становились слишком большими.
- L2 (Ridge): штрафует большие веса, делает модель стабильнее, особенно если признаки похожи друг на друга.
- L1 (Lasso): заставляет часть весов стать ровно 0, выбрасывая часть признаков.
- ElasticNet: сочетает L1 и L2, обычно стабильнее предыдущих и тоже умеет занулять часть весов.

### 1.3 Почему L1 используют для отбора признаков
L1 штраф устроен так, что модель зануляет слабые признаки полностью.
В итоге часть весов становится ровно 0, и эти признаки можно считать неиспользуемыми.

### 1.4 Как теми же моделями учить нелинейные зависимости
Модель остается линейной по весам, но можно сделать признаки нелинейными.
Пример: вместо x подать x, x^2, x^3, x1*x2 и т.д.
Тогда линейная модель на таких признаках сможет описывать нелинейные формы.


###  4. Что такое детерминированная модель и как сделать SGD детерминированным
Детерминированная модель: при одинаковых данных и настройках дает одинаковый результат при каждом запуске.
Чтобы SGD был детерминированным:
- зафиксировать random_state для numpy и для самого алгоритма
- не перемешивать данные случайно, или перемешивать с фиксированным seed
- фиксировать начальные веса через seed, либо делать их одинаковыми, например нулями
- фиксировать порядок батчей и размер батча


### 6. Где нормализация обязательна, а где нет
Нормализация обязательна/полезна:
- линейные модели с регуляризацией (Ridge, Lasso, ElasticNet)
- модели на расстояниях (kNN, k-means)
- градиентные методы, где скорость обучения зависит от масштаба признаков
- когда признаки в разных единицах (метры и миллионы рублей)

Нормализация обычно не нужна в деревьях и всех их проявлениях (лесах и бустингах)

# Preprocessing

In [2]:
train = pd.read_json('data/train.json')
test = pd.read_json('data/test.json')

In [3]:
l, r = train.price.quantile([0.05, 0.95])
train = train.loc[(train.price>=l)&(train.price<=r)]

атата - незя так поступать... но там выбора не было

In [4]:
l, r = test.price.quantile([0.05, 0.95])
test = test.loc[(test.price>=l)&(test.price<=r)]

In [5]:
train.info()

<class 'pandas.DataFrame'>
Index: 44612 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        44612 non-null  float64
 1   bedrooms         44612 non-null  int64  
 2   building_id      44612 non-null  str    
 3   created          44612 non-null  str    
 4   description      44612 non-null  str    
 5   display_address  44612 non-null  str    
 6   features         44612 non-null  object 
 7   latitude         44612 non-null  float64
 8   listing_id       44612 non-null  int64  
 9   longitude        44612 non-null  float64
 10  manager_id       44612 non-null  str    
 11  photos           44612 non-null  object 
 12  price            44612 non-null  int64  
 13  street_address   44612 non-null  str    
 14  interest_level   44612 non-null  str    
dtypes: float64(3), int64(3), object(2), str(7)
memory usage: 5.4+ MB


In [6]:
train.features.head()

4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
Name: features, dtype: object

In [7]:
# features = []
# for string in train.features: 
#     features = features +[elem.strip() for elem in list(string)]
# len(features)

In [8]:
features = [elem.strip() for string in train.features for elem in string ]
len(features)

242337

In [9]:
features = pd.Series(features)
names = features.value_counts().head(20)
names

Elevator               23501
Hardwood Floors        21623
Cats Allowed           21331
Dogs Allowed           19942
Doorman                19003
Dishwasher             18803
No Fee                 16710
Laundry in Building    15042
Fitness Center         12064
Pre-War                 8269
Laundry in Unit         7508
Roof Deck               6046
Outdoor Space           4721
Dining Room             4227
High Speed Internet     3957
Balcony                 2668
Laundry In Building     2430
Swimming Pool           2375
New Construction        2328
Terrace                 1947
Name: count, dtype: int64

In [ ]:
keys = names.index.tolist() 
cols = {f"has_{k}": train.features.map(lambda string, t_k=k: t_k in string).astype("int8") for k in keys}
df = pd.DataFrame()
df = df.assign(**cols)
df

,has_Elevator,has_Hardwood Floors,has_Cats Allowed,has_Dogs Allowed,has_Doorman,has_Dishwasher,has_No Fee,has_Laundry in Building,has_Fitness Center,has_Pre-War,has_Laundry in Unit,has_Roof Deck,has_Outdoor Space,has_Dining Room,has_High Speed Internet,has_Balcony,has_Laundry In Building,has_Swimming Pool,has_New Construction,has_Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
124002,1,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
124004,1,1,1,1,0,1,1,1,0,1,1,0,0,1,0,0,0,0,0,0
124008,0,0,0,0,0,1,1,0,0,1,1,0,1,0,0,0,0,0,0,0


In [11]:
data = pd.concat([df, train.bathrooms.astype('int8'), train.bedrooms.astype('int8')], axis=1)
data.info()

<class 'pandas.DataFrame'>
Index: 44612 entries, 4 to 124009
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   has_Elevator             44612 non-null  int8 
 1   has_Hardwood Floors      44612 non-null  int8 
 2   has_Cats Allowed         44612 non-null  int8 
 3   has_Dogs Allowed         44612 non-null  int8 
 4   has_Doorman              44612 non-null  int8 
 5   has_Dishwasher           44612 non-null  int8 
 6   has_No Fee               44612 non-null  int8 
 7   has_Laundry in Building  44612 non-null  int8 
 8   has_Fitness Center       44612 non-null  int8 
 9   has_Pre-War              44612 non-null  int8 
 10  has_Laundry in Unit      44612 non-null  int8 
 11  has_Roof Deck            44612 non-null  int8 
 12  has_Outdoor Space        44612 non-null  int8 
 13  has_Dining Room          44612 non-null  int8 
 14  has_High Speed Internet  44612 non-null  int8 
 15  has_Balcony      

In [12]:
x_train = data
y_train = train.price
y_train.head()

4     2400
6     3800
9     3495
10    3000
15    2795
Name: price, dtype: int64

#### Препроцессинг для test

In [13]:
test_cols = {f"has_{k}": test.features.map(lambda string, t_k=k: t_k in string).astype("int8") for k in keys}
test_df = pd.DataFrame()
test_df = test_df.assign(**test_cols)
test_data = pd.concat([test_df, test.bathrooms.astype('int8'), test.bedrooms.astype('int8')], axis=1)
x_test = test_data
y_test = test.price
test_data.info()

<class 'pandas.DataFrame'>
Index: 67467 entries, 0 to 124010
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   has_Elevator             67467 non-null  int8 
 1   has_Hardwood Floors      67467 non-null  int8 
 2   has_Cats Allowed         67467 non-null  int8 
 3   has_Dogs Allowed         67467 non-null  int8 
 4   has_Doorman              67467 non-null  int8 
 5   has_Dishwasher           67467 non-null  int8 
 6   has_No Fee               67467 non-null  int8 
 7   has_Laundry in Building  67467 non-null  int8 
 8   has_Fitness Center       67467 non-null  int8 
 9   has_Pre-War              67467 non-null  int8 
 10  has_Laundry in Unit      67467 non-null  int8 
 11  has_Roof Deck            67467 non-null  int8 
 12  has_Outdoor Space        67467 non-null  int8 
 13  has_Dining Room          67467 non-null  int8 
 14  has_High Speed Internet  67467 non-null  int8 
 15  has_Balcony      

# Linear Model implementation

In [ ]:
class my_LinearRegresion:
    def __init__(self, lr = 0.01, iterations = 1000, batch_size = 512):
        self.weights_ = None
        self.bias_ = 0
        self.lr = lr
        self.iterations = iterations
        self.batch_size = batch_size
        self.rng = np.random.default_rng(42)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        l, f = X.shape

        w = np.zeros(f)
        b, lr = self.bias_, self.lr

        for i in range(self.iterations):
            for X_batch, y_batch in self.generate_batches(X, y):
                func = X_batch @ w + b
                error = 2*(func - y_batch)
                desc_w = (X_batch.T @ error) / (2 * len(y_batch))
                desc_b = error.mean()
                w = w - lr * desc_w
                b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self

    def generate_batches(self, X, y):
        assert len(X) == len(y)

        batch_size = self.batch_size
        X = np.array(X)
        y = np.array(y)

        idx = np.arange(len(X))
        self.rng.shuffle(idx)

        for batch_start in range(0, len(X), batch_size):
            batch_idx = idx[batch_start:batch_start+batch_size]
            yield X[batch_idx], y[batch_idx]


    def fit_BGD(self, x, y):
        x = np.asarray(x)
        y = np.asarray(y)
        l, f = x.shape

        w = np.zeros(f)
        b, lr = self.bias_, self.lr

        for i in range(self.iterations):
            func = x @ w + b
            error = 2*(func - y)
            desc_w = (x.T @ error) / l
            desc_b = error.mean()
            w = w - lr * desc_w
            b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self


    def predict(self, x):
        x = np.asarray(x)
        pred = x @ self.weights_ + self.bias_
        return pred

### Metrics implementation

In [15]:
def R2(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    mean = np.mean(y)
    S_error = np.sum((y-pred)**2)
    N_error = np.sum((y-mean)**2)

    if N_error == 0:
        return 1 if S_error == 0 else 0
    R2 = 1 - S_error/N_error
    
    return R2

def MAE(y, pred): 
    return np.mean(np.abs(y - pred))

def RMSE(y, pred):
    return np.sqrt(np.mean((y - pred)**2))


### Fit n Predict

In [16]:
my_lr = my_LinearRegresion()
my_lr.fit(x_train, y_train)

In [17]:
sk_lr = linear_model.SGDRegressor(random_state=42, penalty=None, tol=None)
sk_lr.fit(x_train, y_train)

,"loss loss: str, default='squared_error'The loss function to be used. The possible values are 'squared_error','huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'The 'squared_error' refers to the ordinary least squares fit.'huber' modifies 'squared_error' to focus less on getting outlierscorrect by switching from squared to linear loss past a distance ofepsilon. 'epsilon_insensitive' ignores errors less than epsilon and islinear past that; this is the loss function used in SVR.'squared_epsilon_insensitive' is the same but becomes squared loss pasta tolerance of epsilon.More details about the losses formulas can be found in the:ref:`User Guide `.",'squared_error'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",None
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",None
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42


### MAE

In [18]:
MAE_scores  = pd.DataFrame(columns=["model", "train", "test"])

MAE_scores.loc[len(MAE_scores)] = ["my_lr", MAE(y_train, my_lr.predict(x_train)), MAE(y_test, my_lr.predict(x_test))]
MAE_scores.loc[len(MAE_scores)] = ["sk_lr", MAE(y_train, sk_lr.predict(x_train)), MAE(y_test, sk_lr.predict(x_test))]
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908


### RMSE

In [19]:
RMSE_scores = pd.DataFrame(columns=["model", "train", "test"])

RMSE_scores.loc[len(RMSE_scores)] = ["my_lr", RMSE(y_train, my_lr.predict(x_train)), RMSE(y_test, my_lr.predict(x_test))]
RMSE_scores.loc[len(RMSE_scores)] = ["sk_lr", RMSE(y_train, sk_lr.predict(x_train)), RMSE(y_test, sk_lr.predict(x_test))]
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800


### R2

In [20]:
R2_scores  = pd.DataFrame(columns=["model", "train", "test"])

R2_scores.loc[len(R2_scores)] = ["my_lr", R2(y_train, my_lr.predict(x_train)), R2(y_test, my_lr.predict(x_test))]
R2_scores.loc[len(R2_scores)] = ["sk_lr", R2(y_train, sk_lr.predict(x_train)), R2(y_test, sk_lr.predict(x_test))]
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357


# Ridge, Lasso, ElasticNet implementation

## Lasso

In [21]:
class my_Lasso:

    def __init__(self, alpha = 1, lr = 0.01, iterations = 1000, batch_size = 512):
        self.weights_ = None
        self.bias_ = 0
        self.lr = lr
        self.alpha =alpha
        self.iterations = iterations
        self.batch_size = batch_size
        self.rng = np.random.default_rng(42)


    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        l, f = X.shape

        w = np.zeros(f)
        b, lr, alpha= self.bias_, self.lr, self.alpha

        for i in range(self.iterations):
            for X_batch, y_batch in self.generate_batches(X, y):
                func = X_batch @ w + b
                error = 2*(func - y_batch)
                desc_w = (X_batch.T @ error) / (2* len(y_batch)) + alpha * np.sign(w)
                desc_b = error.mean()

                w = w - lr * desc_w
                b = b - lr * desc_b
    
        self.weights_, self.bias_ = w,b
        return self


    def generate_batches(self, X, y):
        assert len(X) == len(y)

        batch_size = self.batch_size
        X = np.array(X)
        y = np.array(y)

        idx = np.arange(len(X))
        self.rng.shuffle(idx)

        for batch_start in range(0, len(X), batch_size):
            batch_idx = idx[batch_start:batch_start+batch_size]
            yield X[batch_idx], y[batch_idx]


    def fit_BGD(self, x, y):
        x = np.asarray(x)
        y = np.asarray(y)
        l, f = x.shape

        w = np.zeros(f)
        b, lr, alpha = self.bias_, self.lr, self.alpha

        for i in range(self.iterations):
            func = x @ w + b
            error = 2*(func - y)
            desc_w = (x.T @ error) / l
            desc_b = error.mean()

            desc_reg = alpha * np.sign(w)
            desc_w = desc_w + desc_reg

            w = w - lr * desc_w
            b = b - lr * desc_b

        self.weights_, self.bias_ = w,b
        return self


    def predict(self, x):
        x = np.asarray(x)
        pred = x @ self.weights_ + self.bias_
        return pred

In [22]:
my_Lasso = my_Lasso()
my_Lasso.fit(x_train, y_train)

In [23]:
sk_Lasso = linear_model.Lasso(alpha=1)
sk_Lasso.fit(x_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


### MAE

In [24]:
MAE_scores.loc[len(MAE_scores)] = ["my_Lasso", MAE(y_train, my_Lasso.predict(x_train)), MAE(y_test, my_Lasso.predict(x_test))]
MAE_scores.loc[len(MAE_scores)] = ["sk_Lasso", MAE(y_train, sk_Lasso.predict(x_train)), MAE(y_test, sk_Lasso.predict(x_test))]
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908
2,my_Lasso,595.146849,600.411714
3,sk_Lasso,594.303256,599.583917


### RMSE

In [25]:
RMSE_scores.loc[len(RMSE_scores)] = ["my_Lasso", RMSE(y_train, my_Lasso.predict(x_train)), RMSE(y_test, my_Lasso.predict(x_test))]
RMSE_scores.loc[len(RMSE_scores)] = ["sk_Lasso", RMSE(y_train, sk_Lasso.predict(x_train)), RMSE(y_test, sk_Lasso.predict(x_test))]
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767


### R2

In [26]:
R2_scores.loc[len(R2_scores)] = ["my_Lasso", R2(y_train, my_Lasso.predict(x_train)), R2(y_test, my_Lasso.predict(x_test))]
R2_scores.loc[len(R2_scores)] = ["sk_Lasso", R2(y_train, sk_Lasso.predict(x_train)), R2(y_test, sk_Lasso.predict(x_test))]
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117


## Ridge

In [27]:
class my_Ridge:

    def __init__(self, alpha = 0.01, lr = 0.01, iterations = 1000, batch_size = 512):
        self.weights_ = None
        self.bias_ = 0
        self.lr = lr
        self.alpha =alpha
        self.iterations = iterations
        self.batch_size = batch_size
        self.rng = np.random.default_rng(42)


    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        l, f = X.shape

        w = np.zeros(f)
        b, lr, alpha= self.bias_, self.lr, self.alpha

        for i in range(self.iterations):
            for X_batch, y_batch in self.generate_batches(X, y):
                func = X_batch @ w + b
                error = 2*(func - y_batch)
                desc_w = (X_batch.T @ error) / (2 * len(y_batch)) + alpha * w
                desc_b = error.mean()

                w = w - lr * desc_w
                b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self
    

    def generate_batches(self, X, y):
        assert len(X) == len(y)

        batch_size = self.batch_size
        X = np.array(X)
        y = np.array(y)

        idx = np.arange(len(X))
        self.rng.shuffle(idx)

        for batch_start in range(0, len(X), batch_size):
            batch_idx = idx[batch_start:batch_start+batch_size]
            yield X[batch_idx], y[batch_idx]


    def fit_BGD(self, x, y):
        x = np.asarray(x)
        y = np.asarray(y)
        l, f = x.shape

        w = np.zeros(f)
        b, lr, alpha = self.bias_, self.lr, self.alpha

        for i in range(self.iterations):
            func = x @ w + b
            error = 2*(func - y)
            desc_w = (x.T @ error) / l
            desc_b = error.mean()
            desc_w = desc_w + 2 * alpha * w

            w = w - lr * desc_w
            b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self


    def predict(self, x):
        x = np.asarray(x)
        pred = x @ self.weights_ + self.bias_
        return pred

In [28]:
my_Ridge = my_Ridge()
my_Ridge.fit(x_train,y_train)
sk_Ridge = linear_model.Ridge(alpha=0.01)
sk_Ridge.fit(x_train, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.01
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


## MAE

In [29]:
MAE_scores.loc[len(MAE_scores)] = ["my_Ridge", MAE(y_train, my_Ridge.predict(x_train)), MAE(y_test, my_Ridge.predict(x_test))]
MAE_scores.loc[len(MAE_scores)] = ["sk_Ridge", MAE(y_train, sk_Ridge.predict(x_train)), MAE(y_test, sk_Ridge.predict(x_test))]
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908
2,my_Lasso,595.146849,600.411714
3,sk_Lasso,594.303256,599.583917
4,my_Ridge,596.316804,601.409992
5,sk_Ridge,594.060545,599.192951


## RMSE

In [30]:
RMSE_scores.loc[len(RMSE_scores)] = ["my_Ridge", RMSE(y_train, my_Ridge.predict(x_train)), RMSE(y_test, my_Ridge.predict(x_test))]
RMSE_scores.loc[len(RMSE_scores)] = ["sk_Ridge", RMSE(y_train, sk_Ridge.predict(x_train)), RMSE(y_test, sk_Ridge.predict(x_test))]
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145


## R2

In [31]:
R2_scores.loc[len(R2_scores)] = ["my_Ridge", R2(y_train, my_Ridge.predict(x_train)), R2(y_test, my_Ridge.predict(x_test))]
R2_scores.loc[len(R2_scores)] = ["sk_Ridge", R2(y_train, sk_Ridge.predict(x_train)), R2(y_test, sk_Ridge.predict(x_test))]
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117
4,my_Ridge,0.510263,0.396921
5,sk_Ridge,0.511087,0.381856


In [32]:
class my_ElasticNet:

    def __init__(self, alpha = 1, l1_ratio = 0.3, lr = 0.01, iterations = 1000, batch_size = 512):
        self.weights_ = None
        self.bias_ = 0
        self.lr = lr
        self.alpha =alpha
        self.iterations = iterations
        self.batch_size = batch_size
        self.rng = np.random.default_rng(42)
        self.l1_ratio = l1_ratio


    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        l, f = X.shape

        w = np.zeros(f)
        b, lr, alpha, l1_ratio= self.bias_, self.lr, self.alpha, self.l1_ratio

        alpha1 = alpha * l1_ratio
        alpha2 = alpha - alpha1

        for i in range(self.iterations):
            for X_batch, y_batch in self.generate_batches(X, y):
                func = X_batch @ w + b
                error = 2*(func - y_batch)
                desc_w = (X_batch.T @ error) / (2*len(y_batch)) + alpha1 * np.sign(w) +  alpha2 * w
                desc_b = error.mean()

                w = w - lr * desc_w
                b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self


    def generate_batches(self, X, y):
        assert len(X) == len(y)

        batch_size = self.batch_size
        X = np.array(X)
        y = np.array(y)

        idx = np.arange(len(X))
        self.rng.shuffle(idx)

        for batch_start in range(0, len(X), batch_size):
            batch_idx = idx[batch_start:batch_start+batch_size]
            yield X[batch_idx], y[batch_idx]


    def fit_BGD(self, x, y):
        x = np.asarray(x)
        y = np.asarray(y)
        l, f = x.shape

        w = np.zeros(f)
        b, lr, alpha, l1_ratio= self.bias_, self.lr, self.alpha, self.l1_ratio

        alpha1 = alpha * l1_ratio
        alpha2 = alpha - alpha1

        for i in range(self.iterations):
            func = x @ w + b
            error = 2*(func - y)
            desc_w = (x.T @ error) / l
            desc_b = error.mean()
            desc_w = desc_w + + alpha1 * np.sign(w) +  alpha2 * w

            w = w - lr * desc_w
            b = b - lr * desc_b
            
        self.weights_, self.bias_ = w,b
        return self


    def predict(self, x):
        x = np.asarray(x)
        pred = x @ self.weights_ + self.bias_
        return pred

In [33]:
my_ElasticNet = my_ElasticNet(alpha=1, l1_ratio=0.3)
my_ElasticNet.fit(x_train, y_train)
sk_ElasticNet = linear_model.ElasticNet(alpha=1, l1_ratio=0.3)
sk_ElasticNet.fit(x_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",1
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.3
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


## MAE

In [34]:
MAE_scores.loc[len(MAE_scores)] = ["my_ElasticNet", MAE(y_train, my_ElasticNet.predict(x_train)), MAE(y_test, my_ElasticNet.predict(x_test))]
MAE_scores.loc[len(MAE_scores)] = ["sk_ElasticNet", MAE(y_train, sk_ElasticNet.predict(x_train)), MAE(y_test, sk_ElasticNet.predict(x_test))]
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908
2,my_Lasso,595.146849,600.411714
3,sk_Lasso,594.303256,599.583917
4,my_Ridge,596.316804,601.409992
5,sk_Ridge,594.060545,599.192951
6,my_ElasticNet,701.097048,703.729158
7,sk_ElasticNet,699.633142,702.186705


## RMSE

In [35]:
RMSE_scores.loc[len(RMSE_scores)] = ["my_ElasticNet", RMSE(y_train, my_ElasticNet.predict(x_train)), RMSE(y_test, my_ElasticNet.predict(x_test))]
RMSE_scores.loc[len(RMSE_scores)] = ["sk_ElasticNet", RMSE(y_train, sk_ElasticNet.predict(x_train)), RMSE(y_test, sk_ElasticNet.predict(x_test))]
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145
6,my_ElasticNet,892.778446,897.479246
7,sk_ElasticNet,894.535637,899.007504


## R2

In [36]:
R2_scores.loc[len(R2_scores)] = ["my_ElasticNet", R2(y_train, my_ElasticNet.predict(x_train)), R2(y_test, my_ElasticNet.predict(x_test))]
R2_scores.loc[len(R2_scores)] = ["sk_ElasticNet", R2(y_train, sk_ElasticNet.predict(x_train)), R2(y_test, sk_ElasticNet.predict(x_test))]
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117
4,my_Ridge,0.510263,0.396921
5,sk_Ridge,0.511087,0.381856
6,my_ElasticNet,0.369155,0.359521
7,sk_ElasticNet,0.366669,0.357338


# Нормализация

## Математические формулы скейлеров
### MinMaxScaler
X_std = (X - X.min) / (X.max - X.min)  
X_scaled = X_std * (b - a) + a  

X - вектор признака  
X.min - минимальное значение признака
X.max - максимальное значение признака  
X_std - признак отскаллированный в диапазон [0, 1]
X_scaled - признак отскаллированный в заданный диапазон, по дефолту [0, 1]
[a, b] - границы пользовательского интервала  

### StandartScaler
z = (x - u) / s  
x — значение признака у конкретного объекта (одно число)  
u — среднее (mean) этого признака по обучающей выборке  
s — стандартное отклонение (standard deviation) этого признака по обучающей выборке  
s = sqrt( (1/n) · summ ((x − u)^2))  
z — результат стандартизации

In [37]:
class myPreprocessing: 
    def __init__(self):
        pass
    def MinMaxScaler(self, X, columns, custom_range = (0.0, 1.0)):
        scaled_data = pd.DataFrame(index=X.index)
        a, b = custom_range
        for col in columns:
            col_min, col_max = X[col].min(), X[col].max()
            x_range = col_max-col_min
            if x_range == 0 or pd.isna(x_range):
                scaled_data[col] = a
            else:
                scaled_data[col] = (X[col] - col_min) / x_range * (b - a) + a
        return scaled_data
    def StandartScaler(self, X, columns):
        z = pd.DataFrame(index=X.index)
        for col in columns:
            s = np.std(X[col], ddof=0) # ddof чтобы делили на N и сходилось с sklearn
            if s == 0 or pd.isna(s):
                z[col] = 0.0 
            else:
                z[col] = (X[col] - X[col].mean())/s
        return z

In [38]:
scaler = myPreprocessing()
sk_minmax = MinMaxScaler()
sk_standart = StandardScaler()

In [39]:
scaled_x_train = scaler.MinMaxScaler(x_train, columns=x_train.columns)
scaled_x_train.head()

,has_Elevator,has_Hardwood Floors,has_Cats Allowed,has_Dogs Allowed,has_Doorman,has_Dishwasher,has_No Fee,has_Laundry in Building,has_Fitness Center,has_Pre-War,...,has_Outdoor Space,has_Dining Room,has_High Speed Internet,has_Balcony,has_Laundry In Building,has_Swimming Pool,has_New Construction,has_Terrace,bathrooms,bedrooms
4,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.125
6,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.250
9,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.250
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.375
15,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.000


In [40]:
sk_x_train = sk_minmax.fit_transform(X = x_train)
sk_x_train = pd.DataFrame(sk_x_train)
sk_x_train.head()

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.125
1,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.250
2,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.250
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.375
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.000


In [41]:
st_scaled_x_train = scaler.StandartScaler(x_train, x_train.columns)
st_scaled_x_train

,has_Elevator,has_Hardwood Floors,has_Cats Allowed,has_Dogs Allowed,has_Doorman,has_Dishwasher,has_No Fee,has_Laundry in Building,has_Fitness Center,has_Pre-War,...,has_Outdoor Space,has_Dining Room,has_High Speed Internet,has_Balcony,has_Laundry In Building,has_Swimming Pool,has_New Construction,has_Terrace,bathrooms,bedrooms
4,-1.055088,1.031103,1.044709,1.112244,-0.861420,1.171580,-0.773875,1.402080,-0.608812,2.096447,...,-0.344017,3.090964,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,-0.476958
6,0.947788,1.031103,-0.957205,-0.899083,1.160874,1.171580,1.292199,1.402080,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,0.462304
9,0.947788,1.031103,-0.957205,-0.899083,1.160874,1.171580,-0.773875,1.402080,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,0.462304
10,-1.055088,-0.969835,-0.957205,-0.899083,-0.861420,-0.853548,-0.773875,-0.713226,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,1.401566
15,0.947788,-0.969835,-0.957205,-0.899083,1.160874,-0.853548,-0.773875,1.402080,1.642542,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,-1.416220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,0.947788,1.031103,-0.957205,-0.899083,-0.861420,1.171580,-0.773875,-0.713226,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,1.401566
124002,0.947788,-0.969835,1.044709,1.112244,1.160874,-0.853548,1.292199,-0.713226,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,4.166395,-0.237129,-0.234641,-0.213623,-0.382663,0.462304
124004,0.947788,1.031103,1.044709,1.112244,-0.861420,1.171580,1.292199,1.402080,-0.608812,2.096447,...,-0.344017,3.090964,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,-0.476958
124008,-1.055088,-0.969835,-0.957205,-0.899083,-0.861420,1.171580,1.292199,-0.713226,-0.608812,2.096447,...,2.906836,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,0.462304


In [42]:
skst_x_train = sk_standart.fit_transform(x_train)
pd.DataFrame(skst_x_train).head()

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,-1.055088,1.031103,1.044709,1.112244,-0.861420,1.171580,-0.773875,1.402080,-0.608812,2.096447,...,-0.344017,3.090964,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,-0.476958
1,0.947788,1.031103,-0.957205,-0.899083,1.160874,1.171580,1.292199,1.402080,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,0.462304
2,0.947788,1.031103,-0.957205,-0.899083,1.160874,1.171580,-0.773875,1.402080,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,0.462304
3,-1.055088,-0.969835,-0.957205,-0.899083,-0.861420,-0.853548,-0.773875,-0.713226,-0.608812,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,1.401566
4,0.947788,-0.969835,-0.957205,-0.899083,1.160874,-0.853548,-0.773875,1.402080,1.642542,-0.476998,...,-0.344017,-0.323524,-0.311979,-0.252207,-0.240016,-0.237129,-0.234641,-0.213623,-0.382663,-1.416220


In [43]:
scaled_x_test = scaler.MinMaxScaler(x_test, x_test.columns)
st_scaled_x_test = scaler.StandartScaler(x_test, x_test.columns)

# предсказания после стандартизации

## MinMax

In [44]:
models = {
    "my_lr": my_lr,
    "my_Ridge": my_Ridge,
    "my_Lasso": my_Lasso,
    "my_ElasticNet": my_ElasticNet,
}

for name, model in models.items():
    model.fit(scaled_x_train, y_train)

    y_pred_train = model.predict(scaled_x_train)
    y_pred_test = model.predict(scaled_x_test)

    MAE_scores.loc[len(MAE_scores)] = [f"MinMax_{name}", MAE(y_train, y_pred_train), MAE(y_test, y_pred_test)]
    RMSE_scores.loc[len(RMSE_scores)] = [f"MinMax_{name}", RMSE(y_train, y_pred_train), RMSE(y_test, y_pred_test)]
    R2_scores.loc[len(R2_scores)] = [f"MinMax_{name}", R2(y_train, y_pred_train), R2(y_test, y_pred_test)]

In [45]:
y_pred_train = sk_ElasticNet.predict(scaled_x_train)
y_pred_test = sk_ElasticNet.predict(scaled_x_test)
a = [f"MinMax_{name}", MAE(y_train, y_pred_train), MAE(y_test, y_pred_test)]
a

['MinMax_my_ElasticNet',
 np.float64(868.2868111168941),
 np.float64(864.0511119631543)]

In [46]:
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908
2,my_Lasso,595.146849,600.411714
3,sk_Lasso,594.303256,599.583917
4,my_Ridge,596.316804,601.409992
5,sk_Ridge,594.060545,599.192951
6,my_ElasticNet,701.097048,703.729158
7,sk_ElasticNet,699.633142,702.186705
8,MinMax_my_lr,603.465182,683.826811
9,MinMax_my_Ridge,663.750293,663.398592


In [47]:
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145
6,my_ElasticNet,892.778446,897.479246
7,sk_ElasticNet,894.535637,899.007504
8,MinMax_my_lr,793.150707,911.444518
9,MinMax_my_Ridge,858.662689,854.170043


In [48]:
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117
4,my_Ridge,0.510263,0.396921
5,sk_Ridge,0.511087,0.381856
6,my_ElasticNet,0.369155,0.359521
7,sk_ElasticNet,0.366669,0.357338
8,MinMax_my_lr,0.502095,0.339434
9,MinMax_my_Ridge,0.416447,0.419844


## Standart

In [49]:
for name, model in models.items():
    model.fit(st_scaled_x_train, y_train)

    y_pred_train = model.predict(st_scaled_x_train)
    y_pred_test = model.predict(st_scaled_x_test)

    MAE_scores.loc[len(MAE_scores)] = [f"Standart_{name}", MAE(y_train, y_pred_train), MAE(y_test, y_pred_test)]
    RMSE_scores.loc[len(RMSE_scores)] = [f"Standart_{name}", RMSE(y_train, y_pred_train), RMSE(y_test, y_pred_test)]
    R2_scores.loc[len(R2_scores)] = [f"Standart_{name}", R2(y_train, y_pred_train), R2(y_test, y_pred_test)]

In [50]:
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145
6,my_ElasticNet,892.778446,897.479246
7,sk_ElasticNet,894.535637,899.007504
8,MinMax_my_lr,793.150707,911.444518
9,MinMax_my_Ridge,858.662689,854.170043


In [51]:
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145
6,my_ElasticNet,892.778446,897.479246
7,sk_ElasticNet,894.535637,899.007504
8,MinMax_my_lr,793.150707,911.444518
9,MinMax_my_Ridge,858.662689,854.170043


In [52]:
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117
4,my_Ridge,0.510263,0.396921
5,sk_Ridge,0.511087,0.381856
6,my_ElasticNet,0.369155,0.359521
7,sk_ElasticNet,0.366669,0.357338
8,MinMax_my_lr,0.502095,0.339434
9,MinMax_my_Ridge,0.416447,0.419844


# PolyFeatures

In [53]:
from sklearn.preprocessing import PolynomialFeatures

In [54]:
poly = PolynomialFeatures()
poly = PolynomialFeatures(degree=10, interaction_only=True)
poly_train = poly.fit_transform(x_train[["bathrooms", "bedrooms"]])

poly_test = poly.transform(x_test[["bathrooms", "bedrooms"]])
# poly_train = sk_standart.fit_transform(poly_train)
# poly_test  = sk_standart.transform(poly_test)
poly_train

array([[1., 1., 1., 1.],
       [1., 1., 2., 2.],
       [1., 1., 2., 2.],
       ...,
       [1., 1., 1., 1.],
       [1., 1., 2., 2.],
       [1., 1., 3., 3.]], shape=(44612, 4))

In [55]:
pd.DataFrame(poly_train).isna().sum()

0    0
1    0
2    0
3    0
dtype: int64

In [56]:
for name, model in models.items():
    model.fit(poly_train, y_train)

    y_pred_train = model.predict(poly_train)
    y_pred_test = model.predict(poly_test)

    MAE_scores.loc[len(MAE_scores)] = [f"Poly_{name}", MAE(y_train, y_pred_train), MAE(y_test, y_pred_test)]
    RMSE_scores.loc[len(RMSE_scores)] = [f"Poly_{name}", RMSE(y_train, y_pred_train), RMSE(y_test, y_pred_test)]
    R2_scores.loc[len(R2_scores)] = [f"Poly_{name}", R2(y_train, y_pred_train), R2(y_test, y_pred_test)]

# Naive

In [57]:
y_mean, y_median = float(y_train.mean()), float(y_train.median())

baselines = {
    "baseline_mean": y_mean,
    "baseline_median": y_median,
}

tables = {
    "MAE": (MAE_scores, MAE),
    "RMSE": (RMSE_scores, RMSE),
    "R2": (R2_scores, R2),
}

for bname, c in baselines.items():
    tr = np.full(len(y_train), c)
    te = np.full(len(y_test), c)
    for _, (df, metric) in tables.items():
        df.loc[len(df)] = [bname, metric(y_train, tr), metric(y_test, te)]

In [58]:
MAE_scores

,model,train,test
0,my_lr,594.907434,600.012409
1,sk_lr,593.250954,598.410908
2,my_Lasso,595.146849,600.411714
3,sk_Lasso,594.303256,599.583917
4,my_Ridge,596.316804,601.409992
5,sk_Ridge,594.060545,599.192951
6,my_ElasticNet,701.097048,703.729158
7,sk_ElasticNet,699.633142,702.186705
8,MinMax_my_lr,603.465182,683.826811
9,MinMax_my_Ridge,663.750293,663.398592


In [59]:
RMSE_scores

,model,train,test
0,my_lr,785.994621,882.120155
1,sk_lr,786.039632,881.334800
2,my_Lasso,786.176135,881.199352
3,sk_Lasso,786.149790,880.791767
4,my_Ridge,786.617639,870.881448
5,sk_Ridge,785.955878,881.692145
6,my_ElasticNet,892.778446,897.479246
7,sk_ElasticNet,894.535637,899.007504
8,MinMax_my_lr,793.150707,911.444518
9,MinMax_my_Ridge,858.662689,854.170043


In [60]:
R2_scores

,model,train,test
0,my_lr,0.511039,0.381255
1,sk_lr,0.510983,0.382357
2,my_Lasso,0.510813,0.382546
3,sk_Lasso,0.510846,0.383117
4,my_Ridge,0.510263,0.396921
5,sk_Ridge,0.511087,0.381856
6,my_ElasticNet,0.369155,0.359521
7,sk_ElasticNet,0.366669,0.357338
8,MinMax_my_lr,0.502095,0.339434
9,MinMax_my_Ridge,0.416447,0.419844


# FINAL

In [61]:
pd.set_option("display.max_rows", None)
final_scores = pd.concat(
    [
        MAE_scores.assign(metric="MAE"),
        RMSE_scores.assign(metric="RMSE"),
        R2_scores.assign(metric="R2"),
    ],
    ignore_index=True,)[["metric", "model", "train", "test"]]
final_scores

,metric,model,train,test
0,MAE,my_lr,594.907434,600.012409
1,MAE,sk_lr,593.250954,598.410908
2,MAE,my_Lasso,595.146849,600.411714
3,MAE,sk_Lasso,594.303256,599.583917
4,MAE,my_Ridge,596.316804,601.409992
5,MAE,sk_Ridge,594.060545,599.192951
6,MAE,my_ElasticNet,701.097048,703.729158
7,MAE,sk_ElasticNet,699.633142,702.186705
8,MAE,MinMax_my_lr,603.465182,683.826811
9,MAE,MinMax_my_Ridge,663.750293,663.398592


In [62]:

wide = (
    final_scores
    .set_index(["model", "metric"])[["train", "test"]]
    .unstack("metric")
)

wide.columns = wide.columns.swaplevel(0, 1)
wide = wide.sort_index(axis=1, level=0)

wide

metric                         MAE                    R2            \
                              test       train      test     train   
model                                                                
MinMax_my_ElasticNet    846.829994  847.943197  0.099878  0.096800   
MinMax_my_Lasso         669.616592  605.809775  0.363167  0.499094   
MinMax_my_Ridge         663.398592  663.750293  0.419844  0.416447   
MinMax_my_lr            683.826811  603.465182  0.339434  0.502095   
Poly_my_ElasticNet      685.995020  680.500606  0.267852  0.381896   
Poly_my_Lasso           667.019460  660.842284  0.220322  0.408592   
Poly_my_Ridge           668.135128  661.845193  0.216553  0.405895   
Poly_my_lr              666.849591  660.710225  0.221890  0.408715   
Standart_my_ElasticNet  669.725859  646.524391  0.390524  0.452729   
Standart_my_Lasso       610.105440  594.137341  0.441035  0.511004   
Standart_my_Ridge       610.456614  594.326603  0.440968  0.511006   
Standart_my_lr          609.852921  594.050367  0.441145  0.511055   
baseline_mean           890.687588  891.250834 -0.000017  0.000000   
baseline_median         868.521055  869.131467 -0.041316 -0.042787   
my_ElasticNet           703.729158  701.097048  0.359521  0.369155   
my_Lasso                600.411714  595.146849  0.382546  0.510813   
my_Ridge                601.409992  596.316804  0.396921  0.510263   
my_lr                   600.012409  594.907434  0.381255  0.511039   
sk_ElasticNet           702.186705  699.633142  0.357338  0.366669   
sk_Lasso                599.583917  594.303256  0.383117  0.510846   
sk_Ridge                599.192951  594.060545  0.381856  0.511087   
sk_lr                   598.410908  593.250954  0.382357  0.510983   

metric                         RMSE               
                               test        train  
model                                             
MinMax_my_ElasticNet    1063.953432  1068.253142  
MinMax_my_Lasso          894.921134   795.537282  
MinMax_my_Ridge          854.170043   858.662689  
MinMax_my_lr             911.444518   793.150707  
Poly_my_ElasticNet       959.558446   883.716874  
Poly_my_Lasso            990.215494   864.422329  
Poly_my_Ridge            992.605498   866.390949  
Poly_my_lr               989.218856   864.332457  
Standart_my_ElasticNet   875.487888   831.541049  
Standart_my_Lasso        838.424969   786.022646  
Standart_my_Ridge        838.475340   786.020443  
Standart_my_lr           838.342628   785.981387  
baseline_mean           1121.438846  1124.041342  
baseline_median         1144.361683  1147.836613  
my_ElasticNet            897.479246   892.778446  
my_Lasso                 881.199352   786.176135  
my_Ridge                 870.881448   786.617639  
my_lr                    882.120155   785.994621  
sk_ElasticNet            899.007504   894.535637  
sk_Lasso                 880.791767   786.149790  
sk_Ridge                 881.692145   785.955878  
sk_lr                    881.334800   786.039632


### Best model  
- Standart_my_lr (поразительно, обычная линейка)))

### Most stable model
- Standart_my_lr (и тут она хи-хи)

# Additional

In [63]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

my_lr.fit(st_scaled_x_train, y_train_log)

pred_train = np.expm1(my_lr.predict(st_scaled_x_train))
pred_test  = np.expm1(my_lr.predict(st_scaled_x_test))

In [64]:
print("MAE :", MAE(y_train, pred_train),  MAE(y_test, pred_test))
print("RMSE :", RMSE(y_train, pred_train), RMSE(y_test, pred_test))
print("R2  :", R2(y_train, pred_train),   R2(y_test, pred_test))

MAE : 590.7735295662677 153747.41459663626
RMSE : 800.5806943563176 39778016.499064885
R2  : 0.49272243407345495 -1258178449.1387937


### Почему выбросы удаляют только из train, а не из test
Test нужен как честная проверка, на чистых данных
Если удалить выбросы из test, то мы искусственно улучшим оценку качества, что я собсна и сделал)) По сути создав утечку

Ну и логарифмирование таргета здесь не подошло: на train норм, но на test несколько завышенных предсказаний после expm1 вскрылись и убили MAE/RMSE/R2, поэтому предлагаю тоже вскрыться